# 01 - DL / PCA Diagnostic Test

**Purpose:** isolate whether weak DL performance is caused by PCA, by architecture mismatch on static vectors, or by the lack of a non-zero DL contribution in the final ensemble.

**Scope guard:** this notebook is standalone. It does not modify any main source notebook from Phase 1-8 and writes only into diagnostics folders.


## Diagnostic Questions

1. Does PCA hurt the DL branch?
2. Is CNN-BiLSTM-Attention mismatched with static PCA vectors?
3. Can a non-zero DL branch contribute meaningfully to an ensemble?


In [1]:
# try/except: khối xử lý ngoại lệ
try:
    # import google.colab  # type: ignore: import thư viện google
    import google.colab  # type: ignore
    # IN_COLAB: biến cấu hình/hằng số của notebook
    IN_COLAB = True
# except: xử lý ngoại lệ — except Exception:
except Exception:
    # IN_COLAB: biến cấu hình/hằng số của notebook
    IN_COLAB = False

# if: điều kiện — if not IN_COLAB:
if not IN_COLAB:
    # raise RuntimeError('Run this notebook in Google Colab unless local execution is ...: ném lỗi và dừng cell
    raise RuntimeError('Run this notebook in Google Colab unless local execution is explicitly approved.')

# from google.colab import drive: import thư viện google
from google.colab import drive
# drive.mount('/content/drive'): mount Google Drive trên Colab
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# import json: đọc/ghi JSON metadata
import json
# import math: import thư viện math
import math
# import time: đo thời gian thực thi
import time
# from copy import deepcopy: import thư viện copy
from copy import deepcopy
# from pathlib import Path: quản lý đường dẫn
from pathlib import Path

# import numpy: tính toán mảng số
import numpy as np
# import pandas: xử lý DataFrame
import pandas as pd
# import torch: deep learning PyTorch
import torch
# from torch import nn: deep learning PyTorch
from torch import nn
# from torch.utils.data import DataLoader, TensorDataset: deep learning PyTorch
from torch.utils.data import DataLoader, TensorDataset
# from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score: thư viện machine learning scikit-learn
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
# from sklearn.preprocessing import StandardScaler: thư viện machine learning scikit-learn
from sklearn.preprocessing import StandardScaler
# from IPython.display import Markdown, display: import thư viện IPython
from IPython.display import Markdown, display

# SEED: biến cấu hình/hằng số của notebook
SEED = 42
# np.random.seed(SEED): cố định seed numpy
np.random.seed(SEED)
# torch.manual_seed(SEED): cố định seed torch
torch.manual_seed(SEED)
# if: điều kiện — if torch.cuda.is_available():
if torch.cuda.is_available():
    # torch.cuda.manual_seed_all(SEED): thực thi lệnh Python
    torch.cuda.manual_seed_all(SEED)
# torch.backends.cudnn.deterministic = True: lấy giá trị nhỏ nhất
torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = False: thực thi lệnh Python
torch.backends.cudnn.benchmark = False

# ROOT_CANDIDATES: biến cấu hình/hằng số của notebook
ROOT_CANDIDATES = [
    # Path('/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews'),: thực thi lệnh Python
    Path('/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews'),
    # Path('/content/drive/MyDrive/Fake_reviews'),: thực thi lệnh Python
    Path('/content/drive/MyDrive/Fake_reviews'),
# ]: đóng khối danh sách
]
# PROJECT_ROOT: biến cấu hình/hằng số của notebook
PROJECT_ROOT = next((path for path in ROOT_CANDIDATES if path.exists()), ROOT_CANDIDATES[0])
# DATA_DIR: biến cấu hình/hằng số của notebook
DATA_DIR = PROJECT_ROOT / 'data'
# FEATURE_DIR: biến cấu hình/hằng số của notebook
FEATURE_DIR = PROJECT_ROOT / 'artifacts' / 'features'
# PCA_DIR: biến cấu hình/hằng số của notebook
PCA_DIR = PROJECT_ROOT / 'artifacts' / 'pca'
# PRED_DIR: biến cấu hình/hằng số của notebook
PRED_DIR = PROJECT_ROOT / 'artifacts' / 'predictions'
# EVAL_DIR: biến cấu hình/hằng số của notebook
EVAL_DIR = PROJECT_ROOT / 'artifacts' / 'evaluation'
# DIAG_ARTIFACT_DIR: biến cấu hình/hằng số của notebook
DIAG_ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'diagnostics' / 'dl_pca_test'
# DIAG_REPORT_DIR: biến cấu hình/hằng số của notebook
DIAG_REPORT_DIR = PROJECT_ROOT / 'reports' / 'diagnostics' / 'dl_pca_test'
# for: vòng lặp — for path in [DIAG_ARTIFACT_DIR, DIAG_REPORT_DIR]:
for path in [DIAG_ARTIFACT_DIR, DIAG_REPORT_DIR]:
    # path.mkdir(parents=True, exist_ok=True): tạo thư mục nếu chưa có
    path.mkdir(parents=True, exist_ok=True)

# load_array: hàm xử lý load array
def load_array(path: Path):
    # if: điều kiện — if not path.exists():
    if not path.exists():
        # raise FileNotFoundError(f'Missing required artifact: {path}'): ném lỗi và dừng cell
        raise FileNotFoundError(f'Missing required artifact: {path}')
    # return: nạp mảng từ file .npy
    return np.load(path)


# load_csv: hàm xử lý load csv
def load_csv(path: Path):
    # if: điều kiện — if not path.exists():
    if not path.exists():
        # raise FileNotFoundError(f'Missing required artifact: {path}'): ném lỗi và dừng cell
        raise FileNotFoundError(f'Missing required artifact: {path}')
    # return: đọc file CSV vào DataFrame
    return pd.read_csv(path)


# load_json: hàm xử lý load json
def load_json(path: Path):
    # if: điều kiện — if not path.exists():
    if not path.exists():
        # raise FileNotFoundError(f'Missing required artifact: {path}'): ném lỗi và dừng cell
        raise FileNotFoundError(f'Missing required artifact: {path}')
    # with: context manager — with path.open('r', encoding='utf-8') as f:
    with path.open('r', encoding='utf-8') as f:
        # return: parse nội dung JSON
        return json.load(f)


# save_csv: hàm xử lý save csv
def save_csv(df: pd.DataFrame, filename: str):
    # out = ...: đếm số phần tử
    out = DIAG_REPORT_DIR / filename
    # df.to_csv(out, index=False): ghi DataFrame ra file CSV
    df.to_csv(out, index=False)
    # return: trả kết quả từ hàm
    return out


# fmt: hàm xử lý fmt
def fmt(value, digits: int = 6):
    # if: điều kiện — if value is None or (isinstance(value, float) and np.isnan(v
    if value is None or (isinstance(value, float) and np.isnan(value)):
        # return: trả kết quả từ hàm
        return 'NA'
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return f'{float(value):.{digits}f}'
    # except: xử lý ngoại lệ — except Exception:
    except Exception:
        # return: trả kết quả từ hàm
        return str(value)


# metric_bundle: hàm xử lý metric bundle
def metric_bundle(y_true, prob, threshold=0.5):
    # y_pred = ...: ép kiểu dữ liệu cột
    y_pred = (prob >= threshold).astype(int)
    # tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel(): thực thi lệnh Python
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    # out = ...: gán giá trị cho biến out
    out = {
        # 'accuracy': accuracy_score(y_true, y_pred),: thực thi lệnh Python
        'accuracy': accuracy_score(y_true, y_pred),
        # 'macro_f1': f1_score(y_true, y_pred, average='macro'),: thực thi lệnh Python
        'macro_f1': f1_score(y_true, y_pred, average='macro'),
        # 'precision_fake': precision_score(y_true, y_pred, pos_label=1, zero_division=0),: thực thi lệnh Python
        'precision_fake': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        # 'recall_fake': recall_score(y_true, y_pred, pos_label=1, zero_division=0),: thực thi lệnh Python
        'recall_fake': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        # 'f1_fake': f1_score(y_true, y_pred, pos_label=1, zero_division=0),: thực thi lệnh Python
        'f1_fake': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        # 'roc_auc': roc_auc_score(y_true, prob),: thực thi lệnh Python
        'roc_auc': roc_auc_score(y_true, prob),
        # 'pr_auc': average_precision_score(y_true, prob),: thực thi lệnh Python
        'pr_auc': average_precision_score(y_true, prob),
        # 'tn': int(tn),: ép kiểu số nguyên
        'tn': int(tn),
        # 'fp': int(fp),: ép kiểu số nguyên
        'fp': int(fp),
        # 'fn': int(fn),: ép kiểu số nguyên
        'fn': int(fn),
        # 'tp': int(tp),: ép kiểu số nguyên
        'tp': int(tp),
    # }: đóng khối từ điển
    }
    # return: trả kết quả từ hàm
    return out


# choose_threshold_by_precision: hàm xử lý choose threshold by precision
def choose_threshold_by_precision(y_true, prob, target_precision=0.975):
    # grid = ...: gán giá trị cho biến grid
    grid = np.linspace(0.05, 0.95, 181)
    # rows = ...: gán giá trị cho biến rows
    rows = []
    # for: vòng lặp — for threshold in grid:
    for threshold in grid:
        # metrics = ...: ép kiểu số thực
        metrics = metric_bundle(y_true, prob, threshold=float(threshold))
        # rows.append({'threshold': float(threshold), **metrics}): ép kiểu số thực
        rows.append({'threshold': float(threshold), **metrics})
    # df = ...: gán giá trị cho biến df
    df = pd.DataFrame(rows)
    # eligible = ...: gán giá trị cho biến eligible
    eligible = df[df['precision_fake'] >= target_precision]
    # if: điều kiện — if not eligible.empty:
    if not eligible.empty:
        # row = ...: gán giá trị cho biến row
        row = eligible.sort_values(['macro_f1', 'precision_fake', 'roc_auc'], ascending=False).iloc[0]
    # else: nhánh còn lại của điều kiện
    else:
        # row = ...: gán giá trị cho biến row
        row = df.sort_values(['precision_fake', 'macro_f1', 'roc_auc'], ascending=False).iloc[0]
    # return: trả kết quả từ hàm
    return float(row['threshold']), df


# device = ...: chọn thiết bị GPU hoặc CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# BATCH_SIZE: biến cấu hình/hằng số của notebook
BATCH_SIZE = 128
# MAX_EPOCHS: biến cấu hình/hằng số của notebook
MAX_EPOCHS = 15
# PATIENCE: biến cấu hình/hằng số của notebook
PATIENCE = 4
# LEARNING_RATE: biến cấu hình/hằng số của notebook
LEARNING_RATE = 1e-3
# WEIGHT_DECAY: biến cấu hình/hằng số của notebook
WEIGHT_DECAY = 1e-4
# TARGET_PRECISION: biến cấu hình/hằng số của notebook
TARGET_PRECISION = 0.975
# print('PROJECT_ROOT =', PROJECT_ROOT): in thông tin ra console
print('PROJECT_ROOT =', PROJECT_ROOT)
# print('device =', device): in thông tin ra console
print('device =', device)


PROJECT_ROOT = /content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews
device = cpu


In [3]:
# X_raw_train = ...: ép kiểu dữ liệu cột
X_raw_train = load_array(FEATURE_DIR / 'features_raw_train.npy').astype(np.float32)
# X_raw_val = ...: ép kiểu dữ liệu cột
X_raw_val = load_array(FEATURE_DIR / 'features_raw_val.npy').astype(np.float32)
# X_raw_test = ...: ép kiểu dữ liệu cột
X_raw_test = load_array(FEATURE_DIR / 'features_raw_test.npy').astype(np.float32)
# y_train = ...: ép kiểu dữ liệu cột
y_train = load_array(FEATURE_DIR / 'labels_train.npy').astype(np.int64)
# y_val = ...: ép kiểu dữ liệu cột
y_val = load_array(FEATURE_DIR / 'labels_val.npy').astype(np.int64)
# y_test = ...: ép kiểu dữ liệu cột
y_test = load_array(FEATURE_DIR / 'labels_test.npy').astype(np.int64)
# X_pca_train = ...: ép kiểu dữ liệu cột
X_pca_train = load_array(PCA_DIR / 'features_final_train.npy').astype(np.float32)
# X_pca_val = ...: ép kiểu dữ liệu cột
X_pca_val = load_array(PCA_DIR / 'features_final_val.npy').astype(np.float32)
# X_pca_test = ...: ép kiểu dữ liệu cột
X_pca_test = load_array(PCA_DIR / 'features_final_test.npy').astype(np.float32)

# phase7_metrics = ...: gán giá trị cho biến phase7 metrics
phase7_metrics = load_csv(PROJECT_ROOT / 'reports' / 'tables' / 'phase7_final_metrics.csv')
# phase7_ablation = ...: gán giá trị cho biến phase7 ablation
phase7_ablation = load_csv(PROJECT_ROOT / 'reports' / 'tables' / 'phase7_ablation_results.csv')
# phase7_meta = ...: gán giá trị cho biến phase7 meta
phase7_meta = load_json(EVAL_DIR / 'phase7_metadata.json')

# validation_rows = ...: gán giá trị cho biến validation rows
validation_rows = [
    # {'item': 'raw_train_shape', 'value': str(X_raw_train.shape), 'status': 'pass' if...: ép kiểu chuỗi
    {'item': 'raw_train_shape', 'value': str(X_raw_train.shape), 'status': 'pass' if X_raw_train.shape[1] == 777 else 'fail'},
    # {'item': 'raw_val_shape', 'value': str(X_raw_val.shape), 'status': 'pass' if X_r...: ép kiểu chuỗi
    {'item': 'raw_val_shape', 'value': str(X_raw_val.shape), 'status': 'pass' if X_raw_val.shape[1] == 777 else 'fail'},
    # {'item': 'raw_test_shape', 'value': str(X_raw_test.shape), 'status': 'pass' if X...: ép kiểu chuỗi
    {'item': 'raw_test_shape', 'value': str(X_raw_test.shape), 'status': 'pass' if X_raw_test.shape[1] == 777 else 'fail'},
    # {'item': 'pca_train_shape', 'value': str(X_pca_train.shape), 'status': 'pass' if...: ép kiểu chuỗi
    {'item': 'pca_train_shape', 'value': str(X_pca_train.shape), 'status': 'pass' if X_pca_train.shape[1] == 400 else 'fail'},
    # {'item': 'pca_val_shape', 'value': str(X_pca_val.shape), 'status': 'pass' if X_p...: ép kiểu chuỗi
    {'item': 'pca_val_shape', 'value': str(X_pca_val.shape), 'status': 'pass' if X_pca_val.shape[1] == 400 else 'fail'},
    # {'item': 'pca_test_shape', 'value': str(X_pca_test.shape), 'status': 'pass' if X...: ép kiểu chuỗi
    {'item': 'pca_test_shape', 'value': str(X_pca_test.shape), 'status': 'pass' if X_pca_test.shape[1] == 400 else 'fail'},
    # {'item': 'label_train_rows', 'value': str(len(y_train)), 'status': 'pass' if len...: đếm số phần tử
    {'item': 'label_train_rows', 'value': str(len(y_train)), 'status': 'pass' if len(y_train) == X_raw_train.shape[0] else 'fail'},
    # {'item': 'label_val_rows', 'value': str(len(y_val)), 'status': 'pass' if len(y_v...: đếm số phần tử
    {'item': 'label_val_rows', 'value': str(len(y_val)), 'status': 'pass' if len(y_val) == X_raw_val.shape[0] else 'fail'},
    # {'item': 'label_test_rows', 'value': str(len(y_test)), 'status': 'pass' if len(y...: đếm số phần tử
    {'item': 'label_test_rows', 'value': str(len(y_test)), 'status': 'pass' if len(y_test) == X_raw_test.shape[0] else 'fail'},
    # {'item': 'dl_pso_prob_exists', 'value': str((PRED_DIR / 'dl_pso_test_prob.npy')....: kiểm tra file/thư mục tồn tại
    {'item': 'dl_pso_prob_exists', 'value': str((PRED_DIR / 'dl_pso_test_prob.npy').exists()), 'status': 'pass' if (PRED_DIR / 'dl_pso_test_prob.npy').exists() else 'fail'},
    # {'item': 'xgboost_prob_exists', 'value': str((PRED_DIR / 'xgboost_test_prob.npy'...: kiểm tra file/thư mục tồn tại
    {'item': 'xgboost_prob_exists', 'value': str((PRED_DIR / 'xgboost_test_prob.npy').exists()), 'status': 'pass' if (PRED_DIR / 'xgboost_test_prob.npy').exists() else 'fail'},
    # {'item': 'lightgbm_prob_exists', 'value': str((PRED_DIR / 'lightgbm_test_prob.np...: kiểm tra file/thư mục tồn tại
    {'item': 'lightgbm_prob_exists', 'value': str((PRED_DIR / 'lightgbm_test_prob.npy').exists()), 'status': 'pass' if (PRED_DIR / 'lightgbm_test_prob.npy').exists() else 'fail'},
    # {'item': 'final_ensemble_prob_exists', 'value': str((PRED_DIR / 'final_ensemble_...: kiểm tra file/thư mục tồn tại
    {'item': 'final_ensemble_prob_exists', 'value': str((PRED_DIR / 'final_ensemble_test_prob.npy').exists()), 'status': 'pass' if (PRED_DIR / 'final_ensemble_test_prob.npy').exists() else 'fail'},
# ]: đóng khối danh sách
]
# validation_df = ...: gán giá trị cho biến validation df
validation_df = pd.DataFrame(validation_rows)
# save_csv(validation_df, 'input_validation.csv'): thực thi lệnh Python
save_csv(validation_df, 'input_validation.csv')
# display(validation_df): hiển thị bảng/kết quả trên notebook
display(validation_df)

# baseline_rows = ...: gán giá trị cho biến baseline rows
baseline_rows = []
# for: vòng lặp — for model_name, threshold, notes in [
for model_name, threshold, notes in [
    # ('final_ensemble_default_0.5', 0.5, 'Phase 7 final ensemble default threshold'),: thực thi lệnh Python
    ('final_ensemble_default_0.5', 0.5, 'Phase 7 final ensemble default threshold'),
    # ('final_ensemble_selected_0.79', 0.79, 'Phase 7 selected threshold'),: thực thi lệnh Python
    ('final_ensemble_selected_0.79', 0.79, 'Phase 7 selected threshold'),
    # ('lightgbm_test', 0.5, 'Phase 7 best tree baseline'),: thực thi lệnh Python
    ('lightgbm_test', 0.5, 'Phase 7 best tree baseline'),
    # ('dl_pso_test', 0.5, 'Phase 4 DL PSO baseline'),: thực thi lệnh Python
    ('dl_pso_test', 0.5, 'Phase 4 DL PSO baseline'),
# ]:: thực thi lệnh Python
]:
    # if: điều kiện — if model_name.startswith('final_ensemble_default'):
    if model_name.startswith('final_ensemble_default'):
        # row = ...: gán giá trị cho biến row
        row = phase7_metrics[(phase7_metrics['split'] == 'test') & (phase7_metrics['model_variant'] == 'final_ensemble') & (phase7_metrics['threshold'] == 0.5)].iloc[0]
    # elif: nhánh điều kiện phụ — elif model_name.startswith('final_ensemble_selected'):
    elif model_name.startswith('final_ensemble_selected'):
        # row = ...: gán giá trị cho biến row
        row = phase7_metrics[(phase7_metrics['split'] == 'test') & (phase7_metrics['model_variant'] == 'final_ensemble') & (phase7_metrics['threshold'] == 0.79)].iloc[0]
    # elif: nhánh điều kiện phụ — elif model_name == 'lightgbm_test':
    elif model_name == 'lightgbm_test':
        # row = ...: gán giá trị cho biến row
        row = phase7_metrics[(phase7_metrics['split'] == 'test') & (phase7_metrics['model_variant'] == 'lightgbm')].iloc[0]
    # else: nhánh còn lại của điều kiện
    else:
        # row = ...: gán giá trị cho biến row
        row = phase7_metrics[(phase7_metrics['split'] == 'test') & (phase7_metrics['model_variant'] == 'dl_pso')].iloc[0]
    # baseline_rows.append({: thực thi lệnh Python
    baseline_rows.append({
        # 'model': model_name,: thực thi lệnh Python
        'model': model_name,
        # 'macro_f1': row['macro_f1'],: thực thi lệnh Python
        'macro_f1': row['macro_f1'],
        # 'precision_fake': row['precision_fake'],: thực thi lệnh Python
        'precision_fake': row['precision_fake'],
        # 'roc_auc': row['roc_auc'],: thực thi lệnh Python
        'roc_auc': row['roc_auc'],
        # 'notes': notes,: thực thi lệnh Python
        'notes': notes,
    # }): đóng từ điển hoặc DataFrame constructor
    })
# baseline_df = ...: gán giá trị cho biến baseline df
baseline_df = pd.DataFrame(baseline_rows)
# save_csv(baseline_df, 'baseline_snapshot.csv'): thực thi lệnh Python
save_csv(baseline_df, 'baseline_snapshot.csv')
# display(baseline_df): hiển thị bảng/kết quả trên notebook
display(baseline_df)


,item,value,status
0,raw_train_shape,"(29923, 777)",pass
1,raw_val_shape,"(6413, 777)",pass
2,raw_test_shape,"(6413, 777)",pass
3,pca_train_shape,"(29923, 400)",pass
4,pca_val_shape,"(6413, 400)",pass
5,pca_test_shape,"(6413, 400)",pass
6,label_train_rows,29923,pass
7,label_val_rows,6413,pass
8,label_test_rows,6413,pass
9,dl_pso_prob_exists,True,pass


,model,macro_f1,precision_fake,roc_auc,notes
0,final_ensemble_default_0.5,0.855820,0.915566,0.911501,Phase 7 final ensemble default threshold
1,final_ensemble_selected_0.79,0.785982,0.960441,0.911501,Phase 7 selected threshold
2,lightgbm_test,0.860100,0.916900,0.914088,Phase 7 best tree baseline
3,dl_pso_test,0.779349,0.781866,0.851668,Phase 4 DL PSO baseline


In [4]:
# class MLPBinaryClassifier: định nghĩa lớp
class MLPBinaryClassifier(nn.Module):
    # __init__: hàm xử lý   init  
    def __init__(self, input_dim: int, hidden_dims=(256, 128), dropout=0.25):
        # super().__init__(): thực thi lệnh Python
        super().__init__()
        # layers = ...: gán giá trị cho biến layers
        layers = []
        # prev = ...: gán giá trị cho biến prev
        prev = input_dim
        # for: vòng lặp — for hidden in hidden_dims:
        for hidden in hidden_dims:
            # layers.append(nn.Linear(prev, hidden)): thực thi lệnh Python
            layers.append(nn.Linear(prev, hidden))
            # layers.append(nn.BatchNorm1d(hidden)): thực thi lệnh Python
            layers.append(nn.BatchNorm1d(hidden))
            # layers.append(nn.ReLU()): thực thi lệnh Python
            layers.append(nn.ReLU())
            # layers.append(nn.Dropout(dropout)): thực thi lệnh Python
            layers.append(nn.Dropout(dropout))
            # prev = ...: gán giá trị cho biến prev
            prev = hidden
        # layers.append(nn.Linear(prev, 1)): thực thi lệnh Python
        layers.append(nn.Linear(prev, 1))
        # self.net = nn.Sequential(*layers): thực thi lệnh Python
        self.net = nn.Sequential(*layers)

    # forward: hàm xử lý forward
    def forward(self, x):
        # return: trả kết quả từ hàm
        return self.net(x).squeeze(-1)


# class CNNBiLSTMBinaryClassifier: định nghĩa lớp
class CNNBiLSTMBinaryClassifier(nn.Module):
    # __init__: hàm xử lý   init  
    def __init__(self, input_dim: int, conv_channels=48, lstm_hidden=48, dropout=0.25):
        # super().__init__(): thực thi lệnh Python
        super().__init__()
        # self.conv = nn.Sequential(: thực thi lệnh Python
        self.conv = nn.Sequential(
            # nn.Conv1d(1, conv_channels, kernel_size=5, padding=2),: thực thi lệnh Python
            nn.Conv1d(1, conv_channels, kernel_size=5, padding=2),
            # nn.ReLU(),: thực thi lệnh Python
            nn.ReLU(),
            # nn.MaxPool1d(kernel_size=2),: thực thi lệnh Python
            nn.MaxPool1d(kernel_size=2),
            # nn.Dropout(dropout),: thực thi lệnh Python
            nn.Dropout(dropout),
        # ): đóng ngoặc gọi hàm
        )
        # self.lstm = nn.LSTM(: thực thi lệnh Python
        self.lstm = nn.LSTM(
            # input_size = ...: gán giá trị cho biến input size
            input_size=conv_channels,
            # hidden_size = ...: gán giá trị cho biến hidden size
            hidden_size=lstm_hidden,
            # batch_first = ...: gán giá trị cho biến batch first
            batch_first=True,
            # bidirectional = ...: gán giá trị cho biến bidirectional
            bidirectional=True,
        # ): đóng ngoặc gọi hàm
        )
        # self.attn = nn.Sequential(: thực thi lệnh Python
        self.attn = nn.Sequential(
            # nn.Linear(lstm_hidden * 2, lstm_hidden),: thực thi lệnh Python
            nn.Linear(lstm_hidden * 2, lstm_hidden),
            # nn.Tanh(),: thực thi lệnh Python
            nn.Tanh(),
            # nn.Linear(lstm_hidden, 1),: thực thi lệnh Python
            nn.Linear(lstm_hidden, 1),
        # ): đóng ngoặc gọi hàm
        )
        # self.classifier = nn.Sequential(: thực thi lệnh Python
        self.classifier = nn.Sequential(
            # nn.Dropout(dropout),: thực thi lệnh Python
            nn.Dropout(dropout),
            # nn.Linear(lstm_hidden * 2, 1),: thực thi lệnh Python
            nn.Linear(lstm_hidden * 2, 1),
        # ): đóng ngoặc gọi hàm
        )

    # forward: hàm xử lý forward
    def forward(self, x):
        # x = ...: gán giá trị cho biến x
        x = x.unsqueeze(1)
        # x = ...: gán giá trị cho biến x
        x = self.conv(x)
        # x = ...: gán giá trị cho biến x
        x = x.transpose(1, 2)
        # seq, _ = self.lstm(x): thực thi lệnh Python
        seq, _ = self.lstm(x)
        # attn_logits = ...: gán giá trị cho biến attn logits
        attn_logits = self.attn(seq).squeeze(-1)
        # weights = ...: lấy giá trị lớn nhất
        weights = torch.softmax(attn_logits, dim=1).unsqueeze(-1)
        # pooled = ...: tính tổng
        pooled = torch.sum(seq * weights, dim=1)
        # return: trả kết quả từ hàm
        return self.classifier(pooled).squeeze(-1)


# make_loader: tạo PyTorch DataLoader
def make_loader(X, y, batch_size=128, shuffle=False):
    # tensor_x = ...: ép kiểu số thực
    tensor_x = torch.tensor(X, dtype=torch.float32)
    # tensor_y = ...: ép kiểu số thực
    tensor_y = torch.tensor(y, dtype=torch.float32)
    # return: trả kết quả từ hàm
    return DataLoader(TensorDataset(tensor_x, tensor_y), batch_size=batch_size, shuffle=shuffle, drop_last=False)


# predict_probs: hàm xử lý predict probs
def predict_probs(model, X, batch_size=256):
    # model.eval(): thực thi lệnh Python
    model.eval()
    # loader = ...: ép kiểu số thực
    loader = make_loader(X, np.zeros(len(X), dtype=np.float32), batch_size=batch_size, shuffle=False)
    # probs = ...: gán giá trị cho biến probs
    probs = []
    # with: context manager — with torch.no_grad():
    with torch.no_grad():
        # for: vòng lặp — for xb, _ in loader:
        for xb, _ in loader:
            # xb = ...: gán giá trị cho biến xb
            xb = xb.to(device)
            # logits = ...: gán giá trị cho biến logits
            logits = model(xb)
            # probs.append(torch.sigmoid(logits).detach().cpu().numpy()): thực thi lệnh Python
            probs.append(torch.sigmoid(logits).detach().cpu().numpy())
    # return: nối các mảng numpy
    return np.concatenate(probs)


# train_model: hàm xử lý train model
def train_model(model, X_train, y_train, X_val, y_val, X_test, model_name, batch_size=128, max_epochs=15, patience=4, lr=1e-3, weight_decay=1e-4):
    # start = ...: gán giá trị cho biến start
    start = time.perf_counter()
    # model = ...: xóa biến để giải phóng RAM/VRAM
    model = model.to(device)
    # train_loader = ...: gán giá trị cho biến train loader
    train_loader = make_loader(X_train, y_train, batch_size=batch_size, shuffle=True)
    # val_loader = ...: gán giá trị cho biến val loader
    val_loader = make_loader(X_val, y_val, batch_size=batch_size, shuffle=False)
    # optimizer = ...: gán giá trị cho biến optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    # criterion = ...: gán giá trị cho biến criterion
    criterion = nn.BCEWithLogitsLoss()
    # history = ...: gán giá trị cho biến history
    history = []
    # best_state = ...: tạo dictionary
    best_state = deepcopy(model.state_dict())
    # best_score = ...: gán giá trị cho biến best score
    best_score = -1.0
    # best_epoch = ...: gán giá trị cho biến best epoch
    best_epoch = 0
    # patience_counter = ...: gán giá trị cho biến patience counter
    patience_counter = 0
    # for: vòng lặp — for epoch in range(1, max_epochs + 1):
    for epoch in range(1, max_epochs + 1):
        # model.train(): thực thi lệnh Python
        model.train()
        # train_losses = ...: gán giá trị cho biến train losses
        train_losses = []
        # for: vòng lặp — for xb, yb in train_loader:
        for xb, yb in train_loader:
            # xb = ...: gán giá trị cho biến xb
            xb = xb.to(device)
            # yb = ...: gán giá trị cho biến yb
            yb = yb.to(device)
            # optimizer.zero_grad(set_to_none=True): tạo tập hợp
            optimizer.zero_grad(set_to_none=True)
            # logits = ...: gán giá trị cho biến logits
            logits = model(xb)
            # loss = ...: gán giá trị cho biến loss
            loss = criterion(logits, yb)
            # loss.backward(): thực thi lệnh Python
            loss.backward()
            # optimizer.step(): thực thi lệnh Python
            optimizer.step()
            # train_losses.append(float(loss.detach().cpu().item())): ép kiểu số thực
            train_losses.append(float(loss.detach().cpu().item()))
        # model.eval(): thực thi lệnh Python
        model.eval()
        # val_probs = ...: gán giá trị cho biến val probs
        val_probs = []
        # val_losses = ...: gán giá trị cho biến val losses
        val_losses = []
        # with: context manager — with torch.no_grad():
        with torch.no_grad():
            # for: vòng lặp — for xb, yb in val_loader:
            for xb, yb in val_loader:
                # xb = ...: gán giá trị cho biến xb
                xb = xb.to(device)
                # yb = ...: gán giá trị cho biến yb
                yb = yb.to(device)
                # logits = ...: gán giá trị cho biến logits
                logits = model(xb)
                # loss = ...: gán giá trị cho biến loss
                loss = criterion(logits, yb)
                # val_losses.append(float(loss.detach().cpu().item())): ép kiểu số thực
                val_losses.append(float(loss.detach().cpu().item()))
                # val_probs.append(torch.sigmoid(logits).detach().cpu().numpy()): thực thi lệnh Python
                val_probs.append(torch.sigmoid(logits).detach().cpu().numpy())
        # val_probs = ...: nối các mảng numpy
        val_probs = np.concatenate(val_probs)
        # val_metrics = ...: gán giá trị cho biến val metrics
        val_metrics = metric_bundle(y_val, val_probs, threshold=0.5)
        # record = ...: gán giá trị cho biến record
        record = {
            # 'epoch': epoch,: thực thi lệnh Python
            'epoch': epoch,
            # 'train_loss': float(np.mean(train_losses)),: ép kiểu số thực
            'train_loss': float(np.mean(train_losses)),
            # 'val_loss': float(np.mean(val_losses)),: ép kiểu số thực
            'val_loss': float(np.mean(val_losses)),
            # 'val_macro_f1': val_metrics['macro_f1'],: thực thi lệnh Python
            'val_macro_f1': val_metrics['macro_f1'],
            # 'val_precision_fake': val_metrics['precision_fake'],: thực thi lệnh Python
            'val_precision_fake': val_metrics['precision_fake'],
            # 'val_recall_fake': val_metrics['recall_fake'],: thực thi lệnh Python
            'val_recall_fake': val_metrics['recall_fake'],
            # 'val_roc_auc': val_metrics['roc_auc'],: thực thi lệnh Python
            'val_roc_auc': val_metrics['roc_auc'],
        # }: đóng khối từ điển
        }
        # history.append(record): thực thi lệnh Python
        history.append(record)
        # score = ...: gán giá trị cho biến score
        score = val_metrics['macro_f1']
        # if: điều kiện — if score > best_score + 1e-6:
        if score > best_score + 1e-6:
            # best_score = ...: gán giá trị cho biến best score
            best_score = score
            # best_state = ...: tạo dictionary
            best_state = deepcopy(model.state_dict())
            # best_epoch = ...: gán giá trị cho biến best epoch
            best_epoch = epoch
            # patience_counter = ...: gán giá trị cho biến patience counter
            patience_counter = 0
        # else: nhánh còn lại của điều kiện
        else:
            # patience_counter += 1: thực thi lệnh Python
            patience_counter += 1
        # if: điều kiện — if patience_counter >= patience:
        if patience_counter >= patience:
            break
    # model.load_state_dict(best_state): tạo dictionary
    model.load_state_dict(best_state)
    # test_probs = ...: dự đoán nhãn/xác suất
    test_probs = predict_probs(model, X_test, batch_size=batch_size)
    # val_probs = ...: dự đoán nhãn/xác suất
    val_probs = predict_probs(model, X_val, batch_size=batch_size)
    # elapsed = ...: gán giá trị cho biến elapsed
    elapsed = time.perf_counter() - start
    # torch.save(model.state_dict(), DIAG_ARTIFACT_DIR / f'{model_name}.pt'): tạo dictionary
    torch.save(model.state_dict(), DIAG_ARTIFACT_DIR / f'{model_name}.pt')
    # hist_df = ...: gán giá trị cho biến hist df
    hist_df = pd.DataFrame(history)
    # save_csv(hist_df, f'{model_name}_history.csv'): thực thi lệnh Python
    save_csv(hist_df, f'{model_name}_history.csv')
    # selected_t, threshold_grid = choose_threshold_by_precision(y_val, val_probs, tar...: thực thi lệnh Python
    selected_t, threshold_grid = choose_threshold_by_precision(y_val, val_probs, target_precision=TARGET_PRECISION)
    # default_metrics = ...: gán giá trị cho biến default metrics
    default_metrics = metric_bundle(y_test, test_probs, threshold=0.5)
    # selected_metrics = ...: gán giá trị cho biến selected metrics
    selected_metrics = metric_bundle(y_test, test_probs, threshold=selected_t)
    # np.save(DIAG_ARTIFACT_DIR / f'{model_name}_test_prob.npy', test_probs): lưu mảng numpy ra file .npy
    np.save(DIAG_ARTIFACT_DIR / f'{model_name}_test_prob.npy', test_probs)
    # return: trả kết quả từ hàm
    return {
        # 'model_name': model_name,: thực thi lệnh Python
        'model_name': model_name,
        # 'best_epoch': best_epoch,: thực thi lệnh Python
        'best_epoch': best_epoch,
        # 'selected_threshold': selected_t,: thực thi lệnh Python
        'selected_threshold': selected_t,
        # 'elapsed_sec': elapsed,: thực thi lệnh Python
        'elapsed_sec': elapsed,
        # 'history': hist_df,: thực thi lệnh Python
        'history': hist_df,
        # 'threshold_grid': threshold_grid,: thực thi lệnh Python
        'threshold_grid': threshold_grid,
        # 'default_metrics': default_metrics,: thực thi lệnh Python
        'default_metrics': default_metrics,
        # 'selected_metrics': selected_metrics,: thực thi lệnh Python
        'selected_metrics': selected_metrics,
        # 'test_probs': test_probs,: thực thi lệnh Python
        'test_probs': test_probs,
        # 'val_probs': val_probs,: thực thi lệnh Python
        'val_probs': val_probs,
    # }: đóng khối từ điển
    }


# print('model helpers ready'): in thông tin ra console
print('model helpers ready')


model helpers ready


In [5]:
# scaler_raw = ...: gán giá trị cho biến scaler raw
scaler_raw = StandardScaler()
# X_raw_train_s = ...: fit scaler trên train và transform
X_raw_train_s = scaler_raw.fit_transform(X_raw_train).astype(np.float32)
# X_raw_val_s = ...: transform dữ liệu bằng object đã fit
X_raw_val_s = scaler_raw.transform(X_raw_val).astype(np.float32)
# X_raw_test_s = ...: transform dữ liệu bằng object đã fit
X_raw_test_s = scaler_raw.transform(X_raw_test).astype(np.float32)

# scaler_pca = ...: gán giá trị cho biến scaler pca
scaler_pca = StandardScaler()
# X_pca_train_s = ...: fit scaler trên train và transform
X_pca_train_s = scaler_pca.fit_transform(X_pca_train).astype(np.float32)
# X_pca_val_s = ...: transform dữ liệu bằng object đã fit
X_pca_val_s = scaler_pca.transform(X_pca_val).astype(np.float32)
# X_pca_test_s = ...: transform dữ liệu bằng object đã fit
X_pca_test_s = scaler_pca.transform(X_pca_test).astype(np.float32)

# mlp_raw = ...: gán giá trị cho biến mlp raw
mlp_raw = train_model(
    # MLPBinaryClassifier(input_dim=X_raw_train_s.shape[1], hidden_dims=(256, 128), dr...: thực thi lệnh Python
    MLPBinaryClassifier(input_dim=X_raw_train_s.shape[1], hidden_dims=(256, 128), dropout=0.25),
    # X_raw_train_s, y_train,: thực thi lệnh Python
    X_raw_train_s, y_train,
    # X_raw_val_s, y_val,: thực thi lệnh Python
    X_raw_val_s, y_val,
    # X_raw_test_s,: thực thi lệnh Python
    X_raw_test_s,
    # model_name = ...: gán giá trị cho biến model name
    model_name='mlp_raw_777',
    # batch_size = ...: gán giá trị cho biến batch size
    batch_size=BATCH_SIZE,
    # max_epochs = ...: lấy giá trị lớn nhất
    max_epochs=MAX_EPOCHS,
    # patience = ...: gán giá trị cho biến patience
    patience=PATIENCE,
    # lr = ...: gán giá trị cho biến lr
    lr=LEARNING_RATE,
    # weight_decay = ...: gán giá trị cho biến weight decay
    weight_decay=WEIGHT_DECAY,
# ): đóng ngoặc gọi hàm
)

# mlp_pca = ...: gán giá trị cho biến mlp pca
mlp_pca = train_model(
    # MLPBinaryClassifier(input_dim=X_pca_train_s.shape[1], hidden_dims=(256, 128), dr...: thực thi lệnh Python
    MLPBinaryClassifier(input_dim=X_pca_train_s.shape[1], hidden_dims=(256, 128), dropout=0.25),
    # X_pca_train_s, y_train,: thực thi lệnh Python
    X_pca_train_s, y_train,
    # X_pca_val_s, y_val,: thực thi lệnh Python
    X_pca_val_s, y_val,
    # X_pca_test_s,: thực thi lệnh Python
    X_pca_test_s,
    # model_name = ...: gán giá trị cho biến model name
    model_name='mlp_pca_400',
    # batch_size = ...: gán giá trị cho biến batch size
    batch_size=BATCH_SIZE,
    # max_epochs = ...: lấy giá trị lớn nhất
    max_epochs=MAX_EPOCHS,
    # patience = ...: gán giá trị cho biến patience
    patience=PATIENCE,
    # lr = ...: gán giá trị cho biến lr
    lr=LEARNING_RATE,
    # weight_decay = ...: gán giá trị cho biến weight decay
    weight_decay=WEIGHT_DECAY,
# ): đóng ngoặc gọi hàm
)

# mlp_metrics_rows = ...: gán giá trị cho biến mlp metrics rows
mlp_metrics_rows = []
# for: vòng lặp — for tag, result in [('mlp_raw_777', mlp_raw), ('mlp_pca_400'
for tag, result in [('mlp_raw_777', mlp_raw), ('mlp_pca_400', mlp_pca)]:
    # for: vòng lặp — for split_name, probs, labels in [
    for split_name, probs, labels in [
        # ('test_default_0.5', result['test_probs'], y_test),: thực thi lệnh Python
        ('test_default_0.5', result['test_probs'], y_test),
        # ('test_selected_precision', result['test_probs'], y_test),: thực thi lệnh Python
        ('test_selected_precision', result['test_probs'], y_test),
    # ]:: thực thi lệnh Python
    ]:
        # threshold = ...: gán giá trị cho biến threshold
        threshold = 0.5 if split_name == 'test_default_0.5' else result['selected_threshold']
        # metrics = ...: gán giá trị cho biến metrics
        metrics = metric_bundle(labels, probs, threshold=threshold)
        # mlp_metrics_rows.append({: thực thi lệnh Python
        mlp_metrics_rows.append({
            # 'model_variant': tag,: thực thi lệnh Python
            'model_variant': tag,
            # 'split': split_name,: thực thi lệnh Python
            'split': split_name,
            # 'threshold': threshold,: thực thi lệnh Python
            'threshold': threshold,
            # **metrics,: thực thi lệnh Python
            **metrics,
            # 'elapsed_sec': result['elapsed_sec'],: thực thi lệnh Python
            'elapsed_sec': result['elapsed_sec'],
            # 'best_epoch': result['best_epoch'],: thực thi lệnh Python
            'best_epoch': result['best_epoch'],
        # }): đóng từ điển hoặc DataFrame constructor
        })

# mlp_metrics_df = ...: gán giá trị cho biến mlp metrics df
mlp_metrics_df = pd.DataFrame(mlp_metrics_rows)
# save_csv(mlp_metrics_df, 'mlp_raw_vs_pca_metrics.csv'): thực thi lệnh Python
save_csv(mlp_metrics_df, 'mlp_raw_vs_pca_metrics.csv')
# display(mlp_metrics_df): hiển thị bảng/kết quả trên notebook
display(mlp_metrics_df)


,model_variant,split,threshold,accuracy,macro_f1,precision_fake,recall_fake,f1_fake,roc_auc,pr_auc,tn,fp,fn,tp,elapsed_sec,best_epoch
0,mlp_raw_777,test_default_0.5,0.500,0.898487,0.893034,0.921401,0.822027,0.868882,0.949882,0.944606,3605,184,467,2157,27.643424,5
1,mlp_raw_777,test_selected_precision,0.850,0.873226,0.861279,0.974830,0.708460,0.820569,0.949882,0.944606,3741,48,765,1859,27.643424,5
2,mlp_pca_400,test_default_0.5,0.500,0.874630,0.868496,0.878536,0.804878,0.840095,0.931974,0.926860,3497,292,512,2112,12.878595,3
3,mlp_pca_400,test_selected_precision,0.905,0.846562,0.829193,0.970183,0.644817,0.774725,0.931974,0.926860,3737,52,932,1692,12.878595,3


In [6]:
# cnn_pca = ...: gán giá trị cho biến cnn pca
cnn_pca = train_model(
    # CNNBiLSTMBinaryClassifier(input_dim=X_pca_train_s.shape[1], conv_channels=48, ls...: thực thi lệnh Python
    CNNBiLSTMBinaryClassifier(input_dim=X_pca_train_s.shape[1], conv_channels=48, lstm_hidden=48, dropout=0.25),
    # X_pca_train_s, y_train,: thực thi lệnh Python
    X_pca_train_s, y_train,
    # X_pca_val_s, y_val,: thực thi lệnh Python
    X_pca_val_s, y_val,
    # X_pca_test_s,: thực thi lệnh Python
    X_pca_test_s,
    # model_name = ...: gán giá trị cho biến model name
    model_name='cnn_bilstm_pca_400_diagnostic',
    # batch_size = ...: gán giá trị cho biến batch size
    batch_size=BATCH_SIZE,
    # max_epochs = ...: lấy giá trị lớn nhất
    max_epochs=MAX_EPOCHS,
    # patience = ...: gán giá trị cho biến patience
    patience=PATIENCE,
    # lr = ...: gán giá trị cho biến lr
    lr=LEARNING_RATE,
    # weight_decay = ...: gán giá trị cho biến weight decay
    weight_decay=WEIGHT_DECAY,
# ): đóng ngoặc gọi hàm
)

# cnn_metrics_rows = ...: gán giá trị cho biến cnn metrics rows
cnn_metrics_rows = []
# for: vòng lặp — for split_name, threshold in [('test_default_0.5', 0.5), ('t
for split_name, threshold in [('test_default_0.5', 0.5), ('test_selected_precision', cnn_pca['selected_threshold'])]:
    # metrics = ...: gán giá trị cho biến metrics
    metrics = metric_bundle(y_test, cnn_pca['test_probs'], threshold=threshold)
    # cnn_metrics_rows.append({: thực thi lệnh Python
    cnn_metrics_rows.append({
        # 'model_variant': 'cnn_bilstm_pca_400_diagnostic',: thực thi lệnh Python
        'model_variant': 'cnn_bilstm_pca_400_diagnostic',
        # 'split': split_name,: thực thi lệnh Python
        'split': split_name,
        # 'threshold': threshold,: thực thi lệnh Python
        'threshold': threshold,
        # **metrics,: thực thi lệnh Python
        **metrics,
        # 'elapsed_sec': cnn_pca['elapsed_sec'],: thực thi lệnh Python
        'elapsed_sec': cnn_pca['elapsed_sec'],
        # 'best_epoch': cnn_pca['best_epoch'],: thực thi lệnh Python
        'best_epoch': cnn_pca['best_epoch'],
    # }): đóng từ điển hoặc DataFrame constructor
    })

# cnn_metrics_df = ...: gán giá trị cho biến cnn metrics df
cnn_metrics_df = pd.DataFrame(cnn_metrics_rows)
# save_csv(cnn_metrics_df, 'architecture_mismatch_metrics.csv'): thực thi lệnh Python
save_csv(cnn_metrics_df, 'architecture_mismatch_metrics.csv')
# display(cnn_metrics_df): hiển thị bảng/kết quả trên notebook
display(cnn_metrics_df)


,model_variant,split,threshold,accuracy,macro_f1,precision_fake,recall_fake,f1_fake,roc_auc,pr_auc,tn,fp,fn,tp,elapsed_sec,best_epoch
0,cnn_bilstm_pca_400_diagnostic,test_default_0.5,0.500,0.791049,0.774833,0.810445,0.638720,0.714408,0.837894,0.825055,3397,392,948,1676,1679.873553,14
1,cnn_bilstm_pca_400_diagnostic,test_selected_precision,0.935,0.718385,0.645300,0.965831,0.323171,0.484295,0.837894,0.825055,3759,30,1776,848,1679.873553,14


In [7]:
# dl_pso_test = ...: ép kiểu dữ liệu cột
dl_pso_test = load_array(PRED_DIR / 'dl_pso_test_prob.npy').astype(np.float32)
# xgb_test = ...: ép kiểu dữ liệu cột
xgb_test = load_array(PRED_DIR / 'xgboost_test_prob.npy').astype(np.float32)
# lgbm_test = ...: ép kiểu dữ liệu cột
lgbm_test = load_array(PRED_DIR / 'lightgbm_test_prob.npy').astype(np.float32)
# final_test = ...: ép kiểu dữ liệu cột
final_test = load_array(PRED_DIR / 'final_ensemble_test_prob.npy').astype(np.float32)
# dl_pso_val = ...: ép kiểu dữ liệu cột
dl_pso_val = load_array(PRED_DIR / 'dl_pso_val_prob.npy').astype(np.float32)
# xgb_val = ...: ép kiểu dữ liệu cột
xgb_val = load_array(PRED_DIR / 'xgboost_val_prob.npy').astype(np.float32)
# lgbm_val = ...: ép kiểu dữ liệu cột
lgbm_val = load_array(PRED_DIR / 'lightgbm_val_prob.npy').astype(np.float32)

# candidate_prob_map = ...: gán giá trị cho biến candidate prob map
candidate_prob_map = {
    # 'dl_pso': {'val': dl_pso_val, 'test': dl_pso_test},: thực thi lệnh Python
    'dl_pso': {'val': dl_pso_val, 'test': dl_pso_test},
    # 'mlp_raw': {'val': mlp_raw['val_probs'], 'test': mlp_raw['test_probs']},: thực thi lệnh Python
    'mlp_raw': {'val': mlp_raw['val_probs'], 'test': mlp_raw['test_probs']},
    # 'mlp_pca': {'val': mlp_pca['val_probs'], 'test': mlp_pca['test_probs']},: thực thi lệnh Python
    'mlp_pca': {'val': mlp_pca['val_probs'], 'test': mlp_pca['test_probs']},
    # 'cnn_bilstm_pca': {'val': cnn_pca['val_probs'], 'test': cnn_pca['test_probs']},: thực thi lệnh Python
    'cnn_bilstm_pca': {'val': cnn_pca['val_probs'], 'test': cnn_pca['test_probs']},
# }: đóng khối từ điển
}

# sweep_rows = ...: gán giá trị cho biến sweep rows
sweep_rows = []
# dl_weight_grid = ...: gán giá trị cho biến dl weight grid
dl_weight_grid = [0.10, 0.20, 0.30]
# xgb_share_grid = ...: gán giá trị cho biến xgb share grid
xgb_share_grid = [0.00, 0.25, 0.50, 0.75, 1.00]
# for: vòng lặp — for dl_source, probs_bundle in candidate_prob_map.items():
for dl_source, probs_bundle in candidate_prob_map.items():
    # for: vòng lặp — for dl_weight in dl_weight_grid:
    for dl_weight in dl_weight_grid:
        # for: vòng lặp — for xgb_share in xgb_share_grid:
        for xgb_share in xgb_share_grid:
            # remaining = ...: gán giá trị cho biến remaining
            remaining = 1.0 - dl_weight
            # w_xgb = ...: gán giá trị cho biến w xgb
            w_xgb = remaining * xgb_share
            # w_lgbm = ...: gán giá trị cho biến w lgbm
            w_lgbm = remaining * (1.0 - xgb_share)
            # if: điều kiện — if w_xgb < 0 or w_lgbm < 0:
            if w_xgb < 0 or w_lgbm < 0:
                continue
            # val_blend = ...: đếm số phần tử
            val_blend = dl_weight * probs_bundle['val'] + w_xgb * xgb_val + w_lgbm * lgbm_val
            # test_blend = ...: đếm số phần tử
            test_blend = dl_weight * probs_bundle['test'] + w_xgb * xgb_test + w_lgbm * lgbm_test
            # selected_threshold, threshold_grid = choose_threshold_by_precision(y_val, val_bl...: đếm số phần tử
            selected_threshold, threshold_grid = choose_threshold_by_precision(y_val, val_blend, target_precision=TARGET_PRECISION)
            # val_default = ...: đếm số phần tử
            val_default = metric_bundle(y_val, val_blend, threshold=0.5)
            # test_default = ...: đếm số phần tử
            test_default = metric_bundle(y_test, test_blend, threshold=0.5)
            # val_selected = ...: đếm số phần tử
            val_selected = metric_bundle(y_val, val_blend, threshold=selected_threshold)
            # test_selected = ...: đếm số phần tử
            test_selected = metric_bundle(y_test, test_blend, threshold=selected_threshold)
            # sweep_rows.append({: thực thi lệnh Python
            sweep_rows.append({
                # 'candidate_name': f'{dl_source}_dl{dl_weight:.2f}_xgb{w_xgb:.2f}_lgbm{w_lgbm:.2f...: thực thi lệnh Python
                'candidate_name': f'{dl_source}_dl{dl_weight:.2f}_xgb{w_xgb:.2f}_lgbm{w_lgbm:.2f}',
                # 'dl_source': dl_source,: thực thi lệnh Python
                'dl_source': dl_source,
                # 'dl_weight': dl_weight,: thực thi lệnh Python
                'dl_weight': dl_weight,
                # 'xgb_weight': w_xgb,: thực thi lệnh Python
                'xgb_weight': w_xgb,
                # 'lgbm_weight': w_lgbm,: thực thi lệnh Python
                'lgbm_weight': w_lgbm,
                # 'selected_threshold': selected_threshold,: thực thi lệnh Python
                'selected_threshold': selected_threshold,
                # 'val_macro_f1_default': val_default['macro_f1'],: thực thi lệnh Python
                'val_macro_f1_default': val_default['macro_f1'],
                # 'val_precision_fake_default': val_default['precision_fake'],: thực thi lệnh Python
                'val_precision_fake_default': val_default['precision_fake'],
                # 'val_roc_auc_default': val_default['roc_auc'],: thực thi lệnh Python
                'val_roc_auc_default': val_default['roc_auc'],
                # 'val_macro_f1_selected': val_selected['macro_f1'],: thực thi lệnh Python
                'val_macro_f1_selected': val_selected['macro_f1'],
                # 'val_precision_fake_selected': val_selected['precision_fake'],: thực thi lệnh Python
                'val_precision_fake_selected': val_selected['precision_fake'],
                # 'val_roc_auc_selected': val_selected['roc_auc'],: thực thi lệnh Python
                'val_roc_auc_selected': val_selected['roc_auc'],
                # 'test_macro_f1_default': test_default['macro_f1'],: thực thi lệnh Python
                'test_macro_f1_default': test_default['macro_f1'],
                # 'test_precision_fake_default': test_default['precision_fake'],: thực thi lệnh Python
                'test_precision_fake_default': test_default['precision_fake'],
                # 'test_roc_auc_default': test_default['roc_auc'],: thực thi lệnh Python
                'test_roc_auc_default': test_default['roc_auc'],
                # 'test_macro_f1_selected': test_selected['macro_f1'],: thực thi lệnh Python
                'test_macro_f1_selected': test_selected['macro_f1'],
                # 'test_precision_fake_selected': test_selected['precision_fake'],: thực thi lệnh Python
                'test_precision_fake_selected': test_selected['precision_fake'],
                # 'test_roc_auc_selected': test_selected['roc_auc'],: thực thi lệnh Python
                'test_roc_auc_selected': test_selected['roc_auc'],
                # 'delta_vs_phase7_macro_f1_default': test_default['macro_f1'] - 0.855820013913040...: thực thi lệnh Python
                'delta_vs_phase7_macro_f1_default': test_default['macro_f1'] - 0.8558200139130405,
                # 'delta_vs_phase7_precision_fake_default': test_default['precision_fake'] - 0.915...: thực thi lệnh Python
                'delta_vs_phase7_precision_fake_default': test_default['precision_fake'] - 0.9155660377358491,
                # 'delta_vs_phase7_roc_auc_default': test_default['roc_auc'] - 0.9115005769267905,: thực thi lệnh Python
                'delta_vs_phase7_roc_auc_default': test_default['roc_auc'] - 0.9115005769267905,
            # }): đóng từ điển hoặc DataFrame constructor
            })

# sweep_df = ...: gán giá trị cho biến sweep df
sweep_df = pd.DataFrame(sweep_rows)
# save_csv(sweep_df, 'constrained_dl_ensemble_sweep.csv'): ép kiểu chuỗi
save_csv(sweep_df, 'constrained_dl_ensemble_sweep.csv')
# best_candidate = ...: gán giá trị cho biến best candidate
best_candidate = sweep_df.sort_values([
    # 'test_macro_f1_selected',: thực thi lệnh Python
    'test_macro_f1_selected',
    # 'test_precision_fake_selected',: thực thi lệnh Python
    'test_precision_fake_selected',
    # 'test_roc_auc_selected': thực thi lệnh Python
    'test_roc_auc_selected'
# ], ascending=False).iloc[0]: thực thi lệnh Python
], ascending=False).iloc[0]
# best_df = ...: gán giá trị cho biến best df
best_df = pd.DataFrame([best_candidate])
# save_csv(best_df, 'constrained_dl_ensemble_best.csv'): ép kiểu chuỗi
save_csv(best_df, 'constrained_dl_ensemble_best.csv')
# display(sweep_df.head(12)): hiển thị bảng/kết quả trên notebook
display(sweep_df.head(12))
# display(best_df): hiển thị bảng/kết quả trên notebook
display(best_df)


,candidate_name,dl_source,dl_weight,xgb_weight,lgbm_weight,selected_threshold,val_macro_f1_default,val_precision_fake_default,val_roc_auc_default,val_macro_f1_selected,...,val_roc_auc_selected,test_macro_f1_default,test_precision_fake_default,test_roc_auc_default,test_macro_f1_selected,test_precision_fake_selected,test_roc_auc_selected,delta_vs_phase7_macro_f1_default,delta_vs_phase7_precision_fake_default,delta_vs_phase7_roc_auc_default
0,dl_pso_dl0.10_xgb0.00_lgbm0.90,dl_pso,0.1,0.000,0.900,0.800,0.863261,0.921532,0.916377,0.784212,...,0.916377,0.860637,0.917016,0.914499,0.780518,0.963455,0.914499,0.004817,0.001450,0.002999
1,dl_pso_dl0.10_xgb0.23_lgbm0.68,dl_pso,0.1,0.225,0.675,0.795,0.862390,0.920954,0.916387,0.781521,...,0.916387,0.859491,0.917956,0.913534,0.776976,0.965564,0.913534,0.003671,0.002390,0.002034
2,dl_pso_dl0.10_xgb0.45_lgbm0.45,dl_pso,0.1,0.450,0.450,0.795,0.860397,0.920941,0.915688,0.777985,...,0.915688,0.856743,0.915374,0.911969,0.772505,0.963773,0.911969,0.000923,-0.000192,0.000468
3,dl_pso_dl0.10_xgb0.68_lgbm0.23,dl_pso,0.1,0.675,0.225,0.805,0.857473,0.921140,0.914275,0.766559,...,0.914275,0.853040,0.910588,0.909782,0.761322,0.962411,0.909782,-0.002780,-0.004978,-0.001718
4,dl_pso_dl0.10_xgb0.90_lgbm0.00,dl_pso,0.1,0.900,0.000,0.865,0.853711,0.917103,0.912147,0.715190,...,0.912147,0.848179,0.903955,0.906753,0.711516,0.972198,0.906753,-0.007641,-0.011611,-0.004748
5,dl_pso_dl0.20_xgb0.00_lgbm0.80,dl_pso,0.2,0.000,0.800,0.790,0.861305,0.917949,0.916560,0.775686,...,0.916560,0.860995,0.917094,0.914534,0.768308,0.963271,0.914534,0.005175,0.001528,0.003033
6,dl_pso_dl0.20_xgb0.20_lgbm0.60,dl_pso,0.2,0.200,0.600,0.790,0.861233,0.919121,0.916431,0.769102,...,0.916431,0.858513,0.916159,0.913528,0.764107,0.964739,0.913528,0.002693,0.000593,0.002028
7,dl_pso_dl0.20_xgb0.40_lgbm0.40,dl_pso,0.2,0.400,0.400,0.795,0.859214,0.919492,0.915726,0.760733,...,0.915726,0.856484,0.913737,0.912024,0.755135,0.963663,0.912024,0.000664,-0.001830,0.000524
8,dl_pso_dl0.20_xgb0.60_lgbm0.20,dl_pso,0.2,0.600,0.200,0.815,0.856364,0.918483,0.914356,0.742314,...,0.914356,0.852783,0.908963,0.909953,0.734791,0.965436,0.909953,-0.003037,-0.006603,-0.001548
9,dl_pso_dl0.20_xgb0.80_lgbm0.00,dl_pso,0.2,0.800,0.000,0.840,0.854458,0.916865,0.912327,0.716113,...,0.912327,0.848310,0.902062,0.907128,0.710855,0.969644,0.907128,-0.007510,-0.013504,-0.004372


,candidate_name,dl_source,dl_weight,xgb_weight,lgbm_weight,selected_threshold,val_macro_f1_default,val_precision_fake_default,val_roc_auc_default,val_macro_f1_selected,...,val_roc_auc_selected,test_macro_f1_default,test_precision_fake_default,test_roc_auc_default,test_macro_f1_selected,test_precision_fake_selected,test_roc_auc_selected,delta_vs_phase7_macro_f1_default,delta_vs_phase7_precision_fake_default,delta_vs_phase7_roc_auc_default
25,mlp_raw_dl0.30_xgb0.00_lgbm0.70,mlp_raw,0.3,0.0,0.7,0.7,0.881221,0.940932,0.940409,0.853802,...,0.940409,0.879132,0.932907,0.93718,0.847788,0.966165,0.93718,0.023312,0.017341,0.025679


In [8]:
# runtime_rows = ...: gán giá trị cho biến runtime rows
runtime_rows = [
    # {'component': 'mlp_raw_777', 'elapsed_sec': mlp_raw['elapsed_sec']},: thực thi lệnh Python
    {'component': 'mlp_raw_777', 'elapsed_sec': mlp_raw['elapsed_sec']},
    # {'component': 'mlp_pca_400', 'elapsed_sec': mlp_pca['elapsed_sec']},: thực thi lệnh Python
    {'component': 'mlp_pca_400', 'elapsed_sec': mlp_pca['elapsed_sec']},
    # {'component': 'cnn_bilstm_pca_400_diagnostic', 'elapsed_sec': cnn_pca['elapsed_s...: thực thi lệnh Python
    {'component': 'cnn_bilstm_pca_400_diagnostic', 'elapsed_sec': cnn_pca['elapsed_sec']},
    # {'component': 'ensemble_sweep', 'elapsed_sec': float(len(sweep_df))},: ép kiểu số thực
    {'component': 'ensemble_sweep', 'elapsed_sec': float(len(sweep_df))},
# ]: đóng khối danh sách
]
# runtime_df = ...: gán giá trị cho biến runtime df
runtime_df = pd.DataFrame(runtime_rows)
# save_csv(runtime_df, 'runtime_summary.csv'): tính tổng
save_csv(runtime_df, 'runtime_summary.csv')

# decision_rows = ...: gán giá trị cho biến decision rows
decision_rows = [
    # {: thực thi lệnh Python
    {
        # 'question': 'Does PCA hurt DL?',: thực thi lệnh Python
        'question': 'Does PCA hurt DL?',
        # 'evidence': f"MLP raw Macro F1 {fmt(mlp_raw['default_metrics']['macro_f1'])} vs ...: thực thi lệnh Python
        'evidence': f"MLP raw Macro F1 {fmt(mlp_raw['default_metrics']['macro_f1'])} vs MLP PCA Macro F1 {fmt(mlp_pca['default_metrics']['macro_f1'])}",
        # 'rule': 'raw_macro_f1 - pca_macro_f1 >= 0.02',: thực thi lệnh Python
        'rule': 'raw_macro_f1 - pca_macro_f1 >= 0.02',
        # 'decision': 'yes' if (mlp_raw['default_metrics']['macro_f1'] - mlp_pca['default_...: thực thi lệnh Python
        'decision': 'yes' if (mlp_raw['default_metrics']['macro_f1'] - mlp_pca['default_metrics']['macro_f1']) >= 0.02 else 'no_or_inconclusive',
    # },: đóng phần tử từ điển (còn phần tử sau)
    },
    # {: thực thi lệnh Python
    {
        # 'question': 'Is CNN-BiLSTM mismatched with PCA vector?',: thực thi lệnh Python
        'question': 'Is CNN-BiLSTM mismatched with PCA vector?',
        # 'evidence': f"MLP PCA Macro F1 {fmt(mlp_pca['default_metrics']['macro_f1'])} vs ...: thực thi lệnh Python
        'evidence': f"MLP PCA Macro F1 {fmt(mlp_pca['default_metrics']['macro_f1'])} vs CNN-BiLSTM PCA Macro F1 {fmt(cnn_pca['default_metrics']['macro_f1'])}",
        # 'rule': 'mlp_pca_macro_f1 - cnn_pca_macro_f1 >= 0.02',: thực thi lệnh Python
        'rule': 'mlp_pca_macro_f1 - cnn_pca_macro_f1 >= 0.02',
        # 'decision': 'yes' if (mlp_pca['default_metrics']['macro_f1'] - cnn_pca['default_...: thực thi lệnh Python
        'decision': 'yes' if (mlp_pca['default_metrics']['macro_f1'] - cnn_pca['default_metrics']['macro_f1']) >= 0.02 else 'no_or_inconclusive',
    # },: đóng phần tử từ điển (còn phần tử sau)
    },
    # {: thực thi lệnh Python
    {
        # 'question': 'Can DL contribute non-zero to final ensemble?',: thực thi lệnh Python
        'question': 'Can DL contribute non-zero to final ensemble?',
        # 'evidence': f"Best constrained candidate {best_candidate['candidate_name']} with...: ép kiểu chuỗi
        'evidence': f"Best constrained candidate {best_candidate['candidate_name']} with DL weight {fmt(best_candidate['dl_weight'])}",
        # 'rule': 'non_zero_dl within 0.01 Macro F1 / ROC-AUC of phase7 best',: thực thi lệnh Python
        'rule': 'non_zero_dl within 0.01 Macro F1 / ROC-AUC of phase7 best',
        # 'decision': 'yes' if (: thực thi lệnh Python
        'decision': 'yes' if (
            # abs(float(best_candidate['test_macro_f1_selected']) - 0.8558200139130405) <= 0.0...: ép kiểu số thực
            abs(float(best_candidate['test_macro_f1_selected']) - 0.8558200139130405) <= 0.01 and
            # abs(float(best_candidate['test_roc_auc_selected']) - 0.9115005769267905) <= 0.01: ép kiểu số thực
            abs(float(best_candidate['test_roc_auc_selected']) - 0.9115005769267905) <= 0.01
        # ) else 'no_or_inconclusive',: thực thi lệnh Python
        ) else 'no_or_inconclusive',
    # },: đóng phần tử từ điển (còn phần tử sau)
    },
# ]: đóng khối danh sách
]
# decision_df = ...: gán giá trị cho biến decision df
decision_df = pd.DataFrame(decision_rows)
# save_csv(decision_df, 'diagnostic_decision_table.csv'): thực thi lệnh Python
save_csv(decision_df, 'diagnostic_decision_table.csv')
# display(runtime_df): hiển thị bảng/kết quả trên notebook
display(runtime_df)
# display(decision_df): hiển thị bảng/kết quả trên notebook
display(decision_df)

# summary_lines = ...: tính tổng
summary_lines = [
    # ': thực thi lệnh Python
    '# DL / PCA Diagnostic Summary',
    # '',: thực thi lệnh Python
    '',
    # f"- Raw MLP default Macro F1: {fmt(mlp_raw['default_metrics']['macro_f1'])}",: thực thi lệnh Python
    f"- Raw MLP default Macro F1: {fmt(mlp_raw['default_metrics']['macro_f1'])}",
    # f"- PCA MLP default Macro F1: {fmt(mlp_pca['default_metrics']['macro_f1'])}",: thực thi lệnh Python
    f"- PCA MLP default Macro F1: {fmt(mlp_pca['default_metrics']['macro_f1'])}",
    # f"- CNN-BiLSTM on PCA default Macro F1: {fmt(cnn_pca['default_metrics']['macro_f...: thực thi lệnh Python
    f"- CNN-BiLSTM on PCA default Macro F1: {fmt(cnn_pca['default_metrics']['macro_f1'])}",
    # f"- Best constrained non-zero DL blend: {best_candidate['candidate_name']}",: đếm số phần tử
    f"- Best constrained non-zero DL blend: {best_candidate['candidate_name']}",
    # f"- Best constrained blend test Macro F1: {fmt(best_candidate['test_macro_f1_sel...: đếm số phần tử
    f"- Best constrained blend test Macro F1: {fmt(best_candidate['test_macro_f1_selected'])}",
    # f"- Best constrained blend test Precision Fake: {fmt(best_candidate['test_precis...: đếm số phần tử
    f"- Best constrained blend test Precision Fake: {fmt(best_candidate['test_precision_fake_selected'])}",
    # f"- Best constrained blend test ROC-AUC: {fmt(best_candidate['test_roc_auc_selec...: đếm số phần tử
    f"- Best constrained blend test ROC-AUC: {fmt(best_candidate['test_roc_auc_selected'])}",
    # '',: thực thi lệnh Python
    '',
    # ': thực thi lệnh Python
    '## Decision',
    # '',: thực thi lệnh Python
    '',
    # '1. Keep title and rerun with non-zero DL contribution',: thực thi lệnh Python
    '1. Keep title and rerun with non-zero DL contribution',
    # '2. Redesign the DL branch before keeping the title',: thực thi lệnh Python
    '2. Redesign the DL branch before keeping the title',
    # '3. Change title/framing to a tree-based ensemble with DL as ablation',: lấy giá trị nhỏ nhất
    '3. Change title/framing to a tree-based ensemble with DL as ablation',
    # '',: thực thi lệnh Python
    '',
    # 'This diagnostic notebook is meant to decide which of those three directions is ...: tính trung bình
    'This diagnostic notebook is meant to decide which of those three directions is justified by evidence.',
# ]: đóng khối danh sách
]
# (DIAG_REPORT_DIR / 'diagnostic_summary.md').write_text('\n'.join(summary_lines) ...: tính tổng
(DIAG_REPORT_DIR / 'diagnostic_summary.md').write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')
# display(Markdown('\n'.join(summary_lines))): hiển thị bảng/kết quả trên notebook
display(Markdown('\n'.join(summary_lines)))

# final_rows = ...: gán giá trị cho biến final rows
final_rows = []
# final_rows.append({'item': 'raw_minus_pca_macro_f1', 'value': mlp_raw['default_m...: lấy giá trị nhỏ nhất
final_rows.append({'item': 'raw_minus_pca_macro_f1', 'value': mlp_raw['default_metrics']['macro_f1'] - mlp_pca['default_metrics']['macro_f1']})
# final_rows.append({'item': 'mlp_pca_minus_cnn_pca_macro_f1', 'value': mlp_pca['d...: lấy giá trị nhỏ nhất
final_rows.append({'item': 'mlp_pca_minus_cnn_pca_macro_f1', 'value': mlp_pca['default_metrics']['macro_f1'] - cnn_pca['default_metrics']['macro_f1']})
# final_rows.append({'item': 'best_constrained_dl_weight', 'value': float(best_can...: ép kiểu số thực
final_rows.append({'item': 'best_constrained_dl_weight', 'value': float(best_candidate['dl_weight'])})
# final_rows.append({'item': 'best_constrained_macro_f1', 'value': float(best_cand...: ép kiểu số thực
final_rows.append({'item': 'best_constrained_macro_f1', 'value': float(best_candidate['test_macro_f1_selected'])})
# final_rows.append({'item': 'best_constrained_precision_fake', 'value': float(bes...: ép kiểu số thực
final_rows.append({'item': 'best_constrained_precision_fake', 'value': float(best_candidate['test_precision_fake_selected'])})
# final_rows.append({'item': 'best_constrained_roc_auc', 'value': float(best_candi...: ép kiểu số thực
final_rows.append({'item': 'best_constrained_roc_auc', 'value': float(best_candidate['test_roc_auc_selected'])})
# summary_df = ...: tính tổng
summary_df = pd.DataFrame(final_rows)
# save_csv(summary_df, 'diagnostic_summary_table.csv'): tính tổng
save_csv(summary_df, 'diagnostic_summary_table.csv')
# display(summary_df): hiển thị bảng/kết quả trên notebook
display(summary_df)


,component,elapsed_sec
0,mlp_raw_777,27.643424
1,mlp_pca_400,12.878595
2,cnn_bilstm_pca_400_diagnostic,1679.873553
3,ensemble_sweep,60.000000


,question,evidence,rule,decision
0,Does PCA hurt DL?,MLP raw Macro F1 0.893034 vs MLP PCA Macro F1 ...,raw_macro_f1 - pca_macro_f1 >= 0.02,yes
1,Is CNN-BiLSTM mismatched with PCA vector?,MLP PCA Macro F1 0.868496 vs CNN-BiLSTM PCA Ma...,mlp_pca_macro_f1 - cnn_pca_macro_f1 >= 0.02,yes
2,Can DL contribute non-zero to final ensemble?,Best constrained candidate mlp_raw_dl0.30_xgb0...,non_zero_dl within 0.01 Macro F1 / ROC-AUC of ...,no_or_inconclusive


# DL / PCA Diagnostic Summary

- Raw MLP default Macro F1: 0.893034
- PCA MLP default Macro F1: 0.868496
- CNN-BiLSTM on PCA default Macro F1: 0.774833
- Best constrained non-zero DL blend: mlp_raw_dl0.30_xgb0.00_lgbm0.70
- Best constrained blend test Macro F1: 0.847788
- Best constrained blend test Precision Fake: 0.966165
- Best constrained blend test ROC-AUC: 0.937180

## Decision

1. Keep title and rerun with non-zero DL contribution
2. Redesign the DL branch before keeping the title
3. Change title/framing to a tree-based ensemble with DL as ablation

This diagnostic notebook is meant to decide which of those three directions is justified by evidence.

,item,value
0,raw_minus_pca_macro_f1,0.024538
1,mlp_pca_minus_cnn_pca_macro_f1,0.093663
2,best_constrained_dl_weight,0.300000
3,best_constrained_macro_f1,0.847788
4,best_constrained_precision_fake,0.966165
5,best_constrained_roc_auc,0.937180


## What to read from this diagnostic

- If raw MLP clearly beats PCA MLP, PCA is likely removing usable signal.
- If MLP on PCA beats CNN-BiLSTM on PCA, the sequence architecture is likely mismatched with static PCA vectors.
- If no constrained non-zero DL blend stays near the current best, the final title should be changed or the DL branch redesigned.
